# An Induction Head in Disguise: Chasing Grammar in a Character-Level Transformer

I trained a small character level transformer on Nietzsche texts, and now I interpret on it. Numbers can be flattering, but it's the control that breaks, diverts, or makes a conclusion. In trying to understand a particular head in a custom transformer, the conclusion changed frequently. We chased a copying head, quotation grammar, and parenthesis grammar, with the latter two disconfirmed. The takeaway is having identified a copying head with the OV circuit half of a bracket-completion mechanism, but a QK circuit insufficient to route the attention needed to complete it.

## Architecture

The transformer is basically a residual stream with 6 blocks feeding into it, and an embed matrix and unembed matrix on either side. Each block contains 4 attention heads, and a general MLP of Linear, Activation, Linear. This post focuses on just the attention heads, which can be separated into 2 key mechanisms, the QK and OV circuits. The QK circuit is how the head decides what to attend to within the current and previous tokens. The OV circuit decides that given we are attending to a specific token, what token should be boosted when picking the next one.

In [1]:
import math
import random
import numpy
import torch
import torch.nn as nn
from torch.nn import functional as F
from niche_classes import load_model

device = 'cpu'
m = load_model("../niche_model.pt", map_location=device)
print(f"n_layers={m.config.n_layers}  n_head={m.config.n_head}  "
      f"n_embd={m.config.n_embd}  vocab={m.config.vocab_size}")

n_layers=6  n_head=4  n_embd=256  vocab=177


## The copying head

The elusive copying head is the first goal, utilizing the eigenvalue method proposed in the paper "A Mathematical Framework for Transformer Circuits". Iterating through all blocks and heads, we compute the eigenvalues for the OV circuit matrix. A positive eigenvalue means that passing an eigenvector through the OV matrix results in a scaled vector, in the same direction. In other words, it is the same token representation. However the eigenvalues can range to complex numbers as well. So for one head, there can be a mix of complex, positive, and negative eigenvalues. From here, we have to compute a kind of copying score to better characterize whether a head is copying. It is just taking the sum of the head's eigenvalue real parts (up to the matrix's rank, 64 here) and dividing by the sum of their magnitudes. The imaginary part of complex eigenvalues will drive this to 0. B5H0 (Block 5 Head 0) is identified as the only contender.

In [3]:
def independent_weights(attention_block, attention_head):

    w_e = m.model.token_embedding.weight.transpose(-1, -2).detach()                                                                                  # [d_model, vocab_size]
    w_u = m.model.last_layer.weight.detach()                                                                                                         # [vocab_size, d_model]
    qkv = m.model.blocks[attention_block].comp_full_attend.qkv.weight.detach()                                                                       # [3 * d_model, d_model]
    q, k, v = tuple(t.view(m.config.n_head, m.config.n_embd // m.config.n_head, -1).detach() for t in qkv.split(m.config.n_embd))                    # [n_head, d_model / n_head, d_model]
    w_o = torch.stack(m.model.blocks[attention_block].comp_full_attend.lin_proj.weight.split(m.config.n_embd // m.config.n_head, dim=-1)).detach()    # [n_head, d_model, d_model / n_head]

    return w_e, w_u, q, k, v, w_o


def identify_copying_ov(w_u, w_o, v, w_e, attention_block, attention_head):

    w_ov_h = w_u @ w_o[attention_head] @ v[attention_head] @ w_e
    w_ov_h_eigen = torch.linalg.eig(w_ov_h)
    w_ov_h_rank = m.config.n_embd // m.config.n_head

    ov_evalues = w_ov_h_eigen.eigenvalues                                               # compute eigenvalues
    ov_evectors = w_ov_h_eigen.eigenvectors                                             # compute eigenvectors

    ov_evalues_mag = torch.abs(ov_evalues)                                              # magnitudes: torch returns complex evals, magnitude tells significance
    ov_evalues_indsort = torch.argsort(ov_evalues_mag, descending=True)                 # sort eigenvalues by magnitude
    ov_evalues_sorted = ov_evalues[ov_evalues_indsort][:w_ov_h_rank]
    ov_evectors_sorted = ov_evectors[:, ov_evalues_indsort][:, :w_ov_h_rank]            # same sorting for eigenvectors

    # check if the imag part of the top-rank evals are ~0
    ov_evalue_dist_zero = torch.argsort(torch.abs(ov_evalues_sorted.imag))              # sort by imag distance to 0
    ov_ev_sort_by_imag = ov_evalues_sorted[ov_evalue_dist_zero]
    ov_evec_sort_by_imag = ov_evectors_sorted[:, ov_evalue_dist_zero]

    copying_score = sum(ov_evalues_sorted.real) / sum(torch.abs(ov_evalues_sorted))

    return copying_score, ov_ev_sort_by_imag, ov_evec_sort_by_imag

In [5]:
# Sweep every block/head and score the OV circuit's "copying-ness"
for block in range(m.config.n_layers):
    for head in range(m.config.n_head):
        w_e, w_u, q, k, v, w_o = independent_weights(block, head)
        copying, _, _ = identify_copying_ov(w_u, w_o, v, w_e, block, head)
        print(f"Block #{block} Head #{head}: {float(copying.real):.4f}")

Block #0 Head #0: 0.1591
Block #0 Head #1: -0.0052
Block #0 Head #2: 0.2841
Block #0 Head #3: 0.0695
Block #1 Head #0: -0.0004
Block #1 Head #1: 0.3471
Block #1 Head #2: 0.1208
Block #1 Head #3: -0.0500
Block #2 Head #0: -0.0320
Block #2 Head #1: 0.0104
Block #2 Head #2: -0.0342
Block #2 Head #3: -0.0597
Block #3 Head #0: 0.0438
Block #3 Head #1: 0.0575
Block #3 Head #2: 0.0517
Block #3 Head #3: 0.0188
Block #4 Head #0: 0.1179
Block #4 Head #1: 0.0202
Block #4 Head #2: 0.0745
Block #4 Head #3: 0.0733
Block #5 Head #0: 0.6150
Block #5 Head #1: 0.1258
Block #5 Head #2: 0.1358
Block #5 Head #3: 0.0132


With a copying score of 0.615, a spike relative to the others, it proposes the idea that a copying head does exist. Naturally, you would ask what it copies.

## What does it copy?

Eigenvectors are a dead end. They're already in token basis, so it's readable. However, to do that we need to cancel out the norms of the embed and unembed matrices which dominate the reading. But we can't do this with eigenvectors because it mixes all tokens together, there isn't a scalar value to cancel out the norms.

In [6]:
def readout_evectors(ov_evalues, ov_evectors):
    decoded_ev = []
    for i in range(len(ov_evalues)):
        if ov_evalues[i].real > 0 and torch.abs(ov_evalues[i].imag) < 1e-4:
            decoded_ev.append(m.decode(torch.topk(torch.abs(ov_evectors[:, i]), 5).indices))
    return decoded_ev


def readout_block_head(block, head):
    w_e, w_u, q, k, v, w_o = independent_weights(block, head)
    copying_score, ov_ev, ov_evec = identify_copying_ov(w_u, w_o, v, w_e, block, head)
    decoded_ev = readout_evectors(ov_ev, ov_evec)
    return decoded_ev


# Top-5 chars per positive real eigenvector of B5H0's OV circuit -- noisy, no clean signal
for row in readout_block_head(5, 0):
    print(repr(row))

'QJÆ?G'
'gLλyU'
"m'yrê"
'ἀaW"î'
'‐\\κnä'
'Ä?Kί–'
'KÏÄ‐â'
'"“κ\'?'
'"\'”Q—'


So we divert to reading logits directly off the diagonal of the OV circuit. Here we can divide out the norms of the embed and unembed matrices, because the diagonal is per-token, we can compute the norms per token as well. This reveals high logits for many tokens, solidifying its status as a copying head.

In [8]:
def identify_copying_ov_diag(w_u, w_o, v, w_e, attention_block, attention_head):
    w_ov_h = w_u @ w_o[attention_head] @ v[attention_head] @ w_e

    norms = torch.linalg.norm(w_u, dim=1) * torch.linalg.norm(w_e, dim=0)
    diag = (torch.diagonal(w_ov_h) / norms).tolist()

    copying_chars = dict(zip(list(map(m.itos.get, list(range(m.config.vocab_size)))), diag))
    copying_chars = dict(sorted(copying_chars.items(), key=lambda x: x[1], reverse=True))
    return copying_chars


w_e, w_u, q, k, v, w_o = independent_weights(5, 0)
copying_chars = identify_copying_ov_diag(w_u, w_o, v, w_e, 5, 0)

print("Top 10 self-copy logits (attend X -> boost X):")
for ch, val in list(copying_chars.items())[:10]:
    print(f"  {ch!r}: {val:+.4f}")

print("\nCharacters of interest:")
for ch in ['"', '(', ')']:
    print(f"  {ch!r}: {copying_chars[ch]:+.4f}")

Top 10 self-copy logits (attend X -> boost X):
  '"': +0.5945
  'm': +0.4895
  'w': +0.4663
  'l': +0.4471
  'h': +0.4336
  'y': +0.4253
  'g': +0.4242
  'n': +0.4188
  "'": +0.4124
  '-': +0.4070

Characters of interest:
  '"': +0.5945
  '(': +0.2276
  ')': -0.3386


## Quotation grammar?

But beyond that, it had the highest logit for quotes at 0.594. This means that when attending to $"$, the logit for $"$ goes up. Maybe this head knows quotation grammar, the rules are simple enough, every quote just needs to be paired. It just needs to be shown that the QK circuit attends to unpaired $"$ over other characters. So we throw in a couple test sentences and read off the attending tokens from B5H0's attention pattern.

In [10]:
def identify_copying_qk(attention_block, attention_head, sentence, token, iterate):
    chars = [i for i, s in enumerate(sentence) if s == token]
    idx = torch.tensor([[m.stoi[c] for c in sentence]])
    with torch.no_grad():
        m.model(idx, store_attn=True)          # populate

    A = torch.squeeze(m.model.blocks[attention_block].comp_full_attend.attn_weights[:, attention_head])

    q = chars[iterate]                                   # query position
    topk = torch.topk(A[q], 5)
    ranked = [(p, sentence[p], round(w, 4))              # position-keyed, no collisions
              for p, w in zip(topk.indices.tolist(), topk.values.tolist())]
    return q, chars, ranked


def run_case(label, sentence, token, iterate=-1, watch=False):
    q, occs, ranked = identify_copying_qk(5, 0, sentence, token, iterate)
    targets = {p + 1 for p in occs if p < q}             # induction = right after a prior occurrence
    amax_pos, amax_ch, amax_w = ranked[0]                # topk already sorted -> first is argmax
    verdict = ("INDUCTION" if amax_pos in targets else
               "previous-token" if amax_pos == q - 1 else
               "SINK (~pos 0)" if amax_pos <= 1 else "other")

    print("=" * 70)
    print(f"{label}\n  {sentence!r}")
    print(f"  query pos {q} = {sentence[q]!r}   occurrences={occs}   targets={sorted(targets)}")
    print(f"  VERDICT: {verdict}   (top: pos {amax_pos} = {amax_ch!r}, w={amax_w})")
    for p, ch, w in ranked:
        print(f"      pos {p:>3}  {ch!r:>5}  w={w:<7} {'<-- after a prior '+repr(token) if p in targets else ''}")

    if watch:
        A = torch.squeeze(m.model.blocks[5].comp_full_attend.attn_weights[:, 0])
        for wc in watch:
            for p, c in enumerate(sentence):
                if c == wc and p <= q:
                    print(f"      watch {wc!r} @ pos {p:>3}: w={A[q][p].item():.4f}")

    return verdict

In [11]:
# Pair 1 - content swap after the opening quote (induction => top char follows x <-> w)
run_case("P1-A xylophone", 'The man said, "xylophone do I write?". He then spoke, "Grashoper', '"')
run_case("P1-B what",      'The man said, "what do I write?". He then spoke, "Grashoper',       '"')

# Pair 2 - token induction: query the 2nd repeated letter, target = char after the 1st
run_case("P2-A cat", 'the cat sat on the mat, the cat ran', 'c', iterate=1)
run_case("P2-B dog", 'the dog sat on the mat, the dog ran', 'd', iterate=1)

# Pair 3 - induction vs previous-token (A repeats, B doesn't)
run_case("P3-A repeat", 'Zarathustra spoke. Later, Zarathustra', 'Z', iterate=1)
run_case("P3-B norepeat",'Zarathustra spoke to the crowd below',  'Z', iterate=0)  # only one Z

# Pair 4 - matching vs nearest (read the top char: apple=match, banana=nearest)
run_case("P4-A apple-last",  '"apple" ... "banana" ... "apple',  '"')
run_case("P4-B banana-last", '"banana" ... "apple" ... "banana', '"')

# Pair 5 - open vs close quote role
run_case("P5-A prior-opens", 'He said "one. She said "two. He said "', '"')
run_case("P5-B prior-closes",'one" and two" and three" and',          '"')

P1-A xylophone
  'The man said, "xylophone do I write?". He then spoke, "Grashoper'
  query pos 54 = '"'   occurrences=[14, 36, 54]   targets=[15, 37]
  VERDICT: INDUCTION   (top: pos 15 = 'x', w=0.4226)
      pos  15    'x'  w=0.4226  <-- after a prior '"'
      pos  37    '.'  w=0.1272  <-- after a prior '"'
      pos   0    'T'  w=0.1167  
      pos  10    'i'  w=0.0626  
      pos  28    'I'  w=0.0506  
P1-B what
  'The man said, "what do I write?". He then spoke, "Grashoper'
  query pos 49 = '"'   occurrences=[14, 31, 49]   targets=[15, 32]
  VERDICT: INDUCTION   (top: pos 15 = 'w', w=0.6671)
      pos  15    'w'  w=0.6671  <-- after a prior '"'
      pos   0    'T'  w=0.0771  
      pos  32    '.'  w=0.0573  <-- after a prior '"'
      pos  10    'i'  w=0.0392  
      pos  23    'I'  w=0.0226  
P2-A cat
  'the cat sat on the mat, the cat ran'
  query pos 28 = 'c'   occurrences=[4, 28]   targets=[5]
  VERDICT: INDUCTION   (top: pos 5 = 'a', w=0.7721)
      pos   5    'a'  w=0.7721

'SINK (~pos 0)'

But this control reveals the head attends to tokens that follow  previous iterations of the query token. So quote grammar was a dead end, but also proves this head to be an induction head.

## Parenthesis grammar?

Before writing it off completely, the logits for '(' and ')', 0.227 and -0.339 respectively, are also interesting. The OV circuit actually copies '(' and suppresses ')'. It may be able to create opening parenthesis structures, while preventing unnecessary closing structures. This reads as a semblance of parenthesis grammar, but not quite. Checking the logit of ')' when attending to '(' and vice versa could fill in those blanks.

In [13]:
def identify_logit_ov(w_u, w_o, v, w_e, tok1, tok2, attention_block, attention_head):
    w_ov_h = w_u @ w_o[attention_head] @ v[attention_head] @ w_e

    norm_u = torch.linalg.norm(w_u, dim=1)   # per destination token
    norm_e = torch.linalg.norm(w_e, dim=0)   # per source token
    idx = torch.tensor([m.stoi[tok1], m.stoi[tok2]])

    sub   = w_ov_h[idx][:, idx]                          # 2x2: rows=dest, cols=source
    denom = norm_u[idx][:, None] * norm_e[idx][None, :]  # 2x2 outer product
    normed = sub / denom
    return normed


w_e, w_u, q, k, v, w_o = independent_weights(5, 0)
normed = identify_logit_ov(w_u, w_o, v, w_e, '(', ')', 5, 0)
toks = ['(', ')']
for a, t1 in enumerate(toks):
    for b, t2 in enumerate(toks):
        print(f"attend {t2!r} -> logit {t1!r}: {normed[a,b]:+.3f}")

attend '(' -> logit '(': +0.228
attend ')' -> logit '(': -0.105
attend '(' -> logit ')': +0.556
attend ')' -> logit ')': -0.339


Doing so reveals that attending to '(' increases the logit for ')' at 0.556, while attending to ')' decreases the logit for '(' at -0.105.

This could be learned parenthesis grammar. The most notable is the 0.556, when it sees an open parenthesis, it wants to close it. It is tempting, it looks very much like parenthesis grammar. If QK reliably attends to parentheses, and better yet if it can understand parenthesis structure, it is basically what we've been chasing after. So similar to quote analysis, we throw in a couple sentences and check the attending tokens.

In [14]:
# Pair 6 - PRECONDITION: will induction even land on '(' as a successor? (verdict IS valid here)
#   6A target is '(' at pos 1; 6B swaps '(' for neutral 'z'. Compare the WEIGHT on the target.
run_case("P6-A paren-succ",  'x(ab x(cd', 'x')          # occs x@{0,5}, q=5, target={1}='('
run_case("P6-B letter-succ", 'xzab xzcd', 'x')          # occs x@{0,5}, q=5, target={1}='z'

# Pair 7 - THE CRUX: structural attention to an UNMATCHED '(' when closing is due, with NO induction cue.
#   Query = final 's' (unique -> targets empty). '(' sits at pos 3. VERDICT IS MEANINGLESS here.
#   Read A[q][3]. Hypothesis: 7A (unmatched, closing pressure) > 7B (already matched) if completion is live.
run_case("P7-A unmatched", 'So (the cat naps',  's', watch=('(',))    # '(' @3, no ')'
run_case("P7-B matched",   'So (the cat) naps', 's', watch=('(', ')'))# '(' @3, ')' @11

# Case 8 - DISCRIMINATION in one string: unmatched outer '(' @2 vs matched inner '(' @5 (closed by ')' @7).
#   Query = final 'D' (unique). If completion-aware, weight on pos 2 (still open) > pos 5 (already closed).
run_case("C8 open-vs-closed", 'A (B (C) D', 'D', watch=('(', ')'))

P6-A paren-succ
  'x(ab x(cd'
  query pos 5 = 'x'   occurrences=[0, 5]   targets=[1]
  VERDICT: INDUCTION   (top: pos 1 = '(', w=0.4824)
      pos   1    '('  w=0.4824  <-- after a prior 'x'
      pos   0    'x'  w=0.177   
      pos   3    'b'  w=0.1619  
      pos   4    ' '  w=0.0828  
      pos   2    'a'  w=0.0806  
P6-B letter-succ
  'xzab xzcd'
  query pos 5 = 'x'   occurrences=[0, 5]   targets=[1]
  VERDICT: SINK (~pos 0)   (top: pos 0 = 'x', w=0.5706)
      pos   0    'x'  w=0.5706  
      pos   1    'z'  w=0.2403  <-- after a prior 'x'
      pos   4    ' '  w=0.0621  
      pos   5    'x'  w=0.0499  
      pos   2    'a'  w=0.0422  
P7-A unmatched
  'So (the cat naps'
  query pos 15 = 's'   occurrences=[15]   targets=[]
  VERDICT: other   (top: pos 3 = '(', w=0.4017)
      pos   3    '('  w=0.4017  
      pos   7    ' '  w=0.0898  
      pos  11    ' '  w=0.0847  
      pos   0    'S'  w=0.0774  
      pos   5    'h'  w=0.0659  
      watch '(' @ pos   3: w=0.4017
P7-B matche

'previous-token'

And the QK circuit generally does attend to parentheses, but not meaningfully enough to be considered grammar. The head doesn't understand bracket positioning, and just attends to the latest parenthesis. This also holds back the nested grammar that the diagonal suggested, nested parentheses require bracket position understanding, specifically depth tracking to know when to close parentheses.

## Conclusion

As we said initially, this is a copying head with an OV circuit that knows how to handle parentheses, but the QK circuit is blind to bracket structure, and thus can't utilize it properly.